In [ ]:
# 1. Install YOLO (if not already installed)
!pip install ultralytics

import cv2
import os
import glob
import numpy as np
from ultralytics import YOLO

# --- Configuration ---
# We will create an MP4 file
OUTPUT_FILE = 'crowd_density_simulation.mp4'
FPS = 10.0 # Mall dataset is low frame rate, 10 looks natural
# To "simulate" a high-density alert on this specific dataset,
# we set the threshold lower. In a real stadium, this might be 500.
DENSITY_THRESHOLD = 25

# --- 2. Source the Data (Mall Dataset) ---
# If you ran the previous cell, this folder exists. If not, we download it.
if not os.path.exists('/content/datasets/mall_dataset'):
    print("Downloading Mall Dataset...")
    !mkdir -p /content/datasets
    !wget -q http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/mall_dataset.zip -O /content/datasets/mall.zip
    !unzip -q /content/datasets/mall.zip -d /content/datasets/

# Get all frames and sort them
frames = sorted(glob.glob('/content/datasets/mall_dataset/frames/*.jpg'))
print(f"Processing {len(frames)} frames...")

# --- 3. Initialize Model & Video Writer ---
model = YOLO('yolov8n.pt')

# Read first frame to get dimensions
sample_img = cv2.imread(frames[0])
height, width, layers = sample_img.shape
size = (width, height)

# Initialize VideoWriter (mp4v codec works well in Colab)
out = cv2.VideoWriter(OUTPUT_FILE, cv2.VideoWriter_fourcc(*'mp4v'), FPS, size)

# --- 4. Processing Loop ---
# We'll process the first 300 frames (30 seconds) for the demo
for i, frame_path in enumerate(frames[:300]):
    img = cv2.imread(frame_path)

    # Run Detection
    results = model.predict(img, classes=[0], verbose=False)
    count = len(results[0].boxes)

    # --- The "Simulation" Logic ---
    # Visualize the Crowd Density
    if count > DENSITY_THRESHOLD:
        # High Density: Red Theme
        color = (0, 0, 255) # Red
        text = f"CRITICAL DENSITY: {count} PEOPLE"

        # Add a subtle red flashing tint
        overlay = img.copy()
        cv2.rectangle(overlay, (0, 0), (width, height), color, -1)
        img = cv2.addWeighted(overlay, 0.2, img, 0.8, 0)
    else:
        # Normal Density: Green Theme
        color = (0, 255, 0) # Green
        text = f"Density Normal: {count} people"

    # Draw Bounding Boxes
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 1)

    # Draw UI Banner
    cv2.rectangle(img, (0, 0), (width, 50), color, -1)
    cv2.putText(img, text, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

    # Save frame to video
    out.write(img)

    if i % 50 == 0:
        print(f"Processed frame {i}/{len(frames[:300])}...")

# Release resources
out.release()
print(f"Done! Video saved as '{OUTPUT_FILE}'")

# --- 5. Download or Play ---
# Run this to download the MP4 to your local machine
from google.colab import files
files.download(OUTPUT_FILE)

Processing 200 frames starting from frame 1500...
Processed 0/200...
Processed 20/200...


KeyboardInterrupt: 

In [ ]:
import cv2
import os
import glob
import numpy as np
from ultralytics import YOLO

# --- Configuration ---
OUTPUT_FILE = 'smart_crowd_monitor.mp4'
FPS = 10.0
# "Different Clip": We skip the first 1500 frames to see a different time of day
START_FRAME = 1500
DURATION_FRAMES = 200 # Generate 20 seconds of video

# --- Density Thresholds ---
TOTAL_ALARM_LIMIT = 35    # limit for the whole screen
ZONE_ALARM_LIMIT = 10     # limit for a single quadrant (concentrated risk)

# --- 1. Load Data (If not already present) ---
if not os.path.exists('/content/datasets/mall_dataset'):
    print("Downloading Mall Dataset...")
    !mkdir -p /content/datasets
    !wget -q http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/mall_dataset.zip -O /content/datasets/mall.zip
    !unzip -q /content/datasets/mall.zip -d /content/datasets/

frames = sorted(glob.glob('/content/datasets/mall_dataset/frames/*.jpg'))
# Select our "Different Clip" range
selected_frames = frames[START_FRAME : START_FRAME + DURATION_FRAMES]
print(f"Processing {len(selected_frames)} frames starting from frame {START_FRAME}...")

# --- 2. Initialize Model & Video Writer ---
model = YOLO('yolov8n.pt')

# Read dimensions
sample_img = cv2.imread(frames[0])
H, W, _ = sample_img.shape

out = cv2.VideoWriter(OUTPUT_FILE, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))

# --- 3. Define the Zones (Quadrants) ---
# We split the image into 4 zones: Top-Left, Top-Right, Bottom-Left, Bottom-Right
# Format: (x_start, y_start, x_end, y_end)
zones = [
    (0, 0, W//2, H//2),       # Zone 1: Top-Left
    (W//2, 0, W, H//2),       # Zone 2: Top-Right
    (0, H//2, W//2, H),       # Zone 3: Bottom-Left
    (W//2, H//2, W, H)        # Zone 4: Bottom-Right
]

def check_zone_density(center_x, center_y):
    """Returns the index of the zone (0-3) a person belongs to."""
    col = 0 if center_x < W // 2 else 1
    row = 0 if center_y < H // 2 else 1
    return row * 2 + col

# --- 4. Main Processing Loop ---
for i, frame_path in enumerate(selected_frames):
    img = cv2.imread(frame_path)
    overlay = img.copy() # For transparent highlights

    # Run YOLO
    results = model.predict(img, classes=[0], verbose=False)

    # Initialize Zone Counters
    zone_counts = [0, 0, 0, 0]

    # Process Detections
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)

        # Calculate "Center" of the person (better for zone accuracy)
        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2

        # Add to zone count
        z_idx = check_zone_density(cx, cy)
        zone_counts[z_idx] += 1

        # Draw standard person box (Green)
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)

    # --- INTELLIGENT WARNING SYSTEM ---

    # Check Each Zone for "Concentration"
    for z_idx, count in enumerate(zone_counts):
        if count > ZONE_ALARM_LIMIT:
            # Get zone coordinates
            zx1, zy1, zx2, zy2 = zones[z_idx]

            # Draw THICK ORANGE BOX around the high-concentration zone
            cv2.rectangle(img, (zx1, zy1), (zx2, zy2), (0, 165, 255), 4)

            # Add Zone Label
            cv2.putText(img, f"ZONE {z_idx+1} HIGH: {count}", (zx1+10, zy1+30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)

    # Check Total Screen Density
    total_count = sum(zone_counts)
    if total_count > TOTAL_ALARM_LIMIT:
        # Flash Red Overlay
        cv2.rectangle(overlay, (0, 0), (W, H), (0, 0, 255), -1)
        img = cv2.addWeighted(overlay, 0.2, img, 0.8, 0)
        status = f"CRITICAL: {total_count} (Total)"
        header_color = (0, 0, 255)
    else:
        status = f"Normal: {total_count} (Total)"
        header_color = (0, 255, 0)

    # Top Status Bar
    cv2.rectangle(img, (0, 0), (W, 40), header_color, -1)
    cv2.putText(img, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

    # Draw Grid Lines (to make zones visible)
    cv2.line(img, (W//2, 0), (W//2, H), (255, 255, 255), 1)
    cv2.line(img, (0, H//2), (W, H//2), (255, 255, 255), 1)

    out.write(img)
    if i % 20 == 0:
        print(f"Processed {i}/{DURATION_FRAMES}...")

out.release()
print(f"Video saved as {OUTPUT_FILE}")

# --- 5. Download the file ---
from google.colab import files
files.download(OUTPUT_FILE)

Processing 200 frames starting from frame 1500...
Processed 0/200...
Processed 20/200...
Processed 40/200...
Processed 60/200...
Processed 80/200...
Processed 100/200...
Processed 120/200...
Processed 140/200...
Processed 160/200...
Processed 180/200...
Video saved as smart_crowd_monitor.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import os
import glob
import numpy as np
from ultralytics import YOLO

# --- Configuration ---
OUTPUT_FILE = 'amenity_monitor.mp4'
FPS = 10.0
START_FRAME = 800  # Mid-day traffic
DURATION_FRAMES = 200

# --- Amenity Logic ---
# We define a polygon for a "Promotional Stand" on the left side of the hallway
AMENITY_NAME = "West Wing Promo Stand"
AMENITY_LIMIT = 6 # If more than 6 people are INSIDE this small area -> Risk.
RECOMMENDED_ACTION = "Dispatch floor manager to redirect flow."

# Coordinates for the 'Virtual' Stand (x, y points)
# These points map roughly to the left-center floor area in the Mall Dataset
AMENITY_ZONE = np.array([
    [10, 250],   # Top Left
    [200, 250],  # Top Right
    [200, 480],  # Bottom Right
    [10, 480]    # Bottom Left
], np.int32)

# --- 1. Load Data ---
if not os.path.exists('/content/datasets/mall_dataset'):
    print("Downloading Mall Dataset...")
    !mkdir -p /content/datasets
    !wget -q http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/mall_dataset.zip -O /content/datasets/mall.zip
    !unzip -q /content/datasets/mall.zip -d /content/datasets/

frames = sorted(glob.glob('/content/datasets/mall_dataset/frames/*.jpg'))
selected_frames = frames[START_FRAME : START_FRAME + DURATION_FRAMES]

# --- 2. Setup Video Writer ---
model = YOLO('yolov8n.pt')
sample_img = cv2.imread(frames[0])
H, W, _ = sample_img.shape
out = cv2.VideoWriter(OUTPUT_FILE, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))

# --- Helper: Check if point is in polygon ---
def is_in_zone(point, polygon):
    # pointPolygonTest returns >= 0 if point is inside or on edge
    return cv2.pointPolygonTest(polygon, point, False) >= 0

print(f"Monitoring '{AMENITY_NAME}' for crowding...")

# --- 3. Main Loop ---
for i, frame_path in enumerate(selected_frames):
    img = cv2.imread(frame_path)
    overlay = img.copy()

    # Run YOLO
    results = model.predict(img, classes=[0], verbose=False)

    amenity_count = 0

    # Process Each Person
    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)

        # Calculate FEET position (Bottom Center of the box)
        # This is crucial: we care where they stand, not where their head is.
        feet_x = (x1 + x2) // 2
        feet_y = y2
        feet_point = (feet_x, feet_y)

        # Check if they are inside the specific Amenity Zone
        if is_in_zone(feet_point, AMENITY_ZONE):
            amenity_count += 1
            # Draw person in RED if they are contributing to the congestion
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)
        else:
            # Draw person in Green if they are just passing by
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 1)

    # --- Draw the Amenity Zone on the Floor ---
    # Logic: Green floor = Safe, Red floor = Crowded
    if amenity_count > AMENITY_LIMIT:
        zone_color = (0, 0, 255) # Red
        alert_active = True
    else:
        zone_color = (255, 255, 0) # Cyan/Yellowish
        alert_active = False

    # Draw transparent polygon on the floor
    cv2.fillPoly(overlay, [AMENITY_ZONE], zone_color)
    img = cv2.addWeighted(overlay, 0.3, img, 0.7, 0)

    # Draw Border around zone
    cv2.polylines(img, [AMENITY_ZONE], True, zone_color, 2)

    # Label the Zone
    cv2.putText(img, f"{AMENITY_NAME}: {amenity_count}", (20, 460),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # --- THE POPUP LOGIC ---
    if alert_active:
        # Create a "Popup Window" style graphic
        # White background box with black border
        popup_x, popup_y = 50, 50
        popup_w, popup_h = 540, 120

        # Draw Popup Background
        cv2.rectangle(img, (popup_x, popup_y), (popup_x + popup_w, popup_y + popup_h), (240, 240, 240), -1)
        cv2.rectangle(img, (popup_x, popup_y), (popup_x + popup_w, popup_y + popup_h), (50, 50, 50), 2)

        # Draw Warning Icon (Red Circle)
        cv2.circle(img, (popup_x + 40, popup_y + 40), 20, (0, 0, 255), -1)
        cv2.putText(img, "!", (popup_x + 35, popup_y + 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)

        # Draw Text Lines
        # Title
        cv2.putText(img, f"ALERT: Crowding around {AMENITY_NAME}",
                    (popup_x + 80, popup_y + 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

        # Action Item (Smaller font)
        cv2.putText(img, f"Rec. Action: {RECOMMENDED_ACTION}",
                    (popup_x + 20, popup_y + 90), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (50, 50, 50), 1)

    out.write(img)
    if i % 25 == 0:
        print(f"Processed frame {i}/{DURATION_FRAMES}...")

out.release()
print(f"Video saved as {OUTPUT_FILE}")

# --- Download ---
from google.colab import files
files.download(OUTPUT_FILE)

Monitoring 'West Wing Promo Stand' for crowding...
Processed frame 0/200...
Processed frame 25/200...
Processed frame 50/200...
Processed frame 75/200...
Processed frame 100/200...
Processed frame 125/200...
Processed frame 150/200...
Processed frame 175/200...
Video saved as amenity_monitor.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import cv2
import os
import glob
import numpy as np
from google.colab import files

# --- Configuration ---
OUTPUT_FILE = 'medical_emergency_response_fixed.mp4'
FPS = 10.0
FRAME_INDEX = 1650  # A frame with people sitting on benches
DURATION_FRAMES = 100 # 10 seconds of video showing the alert

# --- CORRECTED Emergency Data ---
# New coordinates to frame the person on the bench
EMERGENCY_BOX = (180, 320, 260, 450) # (x1, y1, x2, y2)
LOCATION_NAME = "North Corridor - Bench Area B"
INCIDENT_TYPE = "SUSPECTED MEDICAL EMERGENCY"
ACTION_PLAN = "DISPATCH ON-SITE MEDICAL TEAM"

# --- 1. Load Data ---
if not os.path.exists('/content/datasets/mall_dataset'):
    print("Downloading Mall Dataset...")
    !mkdir -p /content/datasets
    !wget -q http://personal.ie.cuhk.edu.hk/~ccloy/files/datasets/mall_dataset.zip -O /content/datasets/mall.zip
    !unzip -q /content/datasets/mall.zip -d /content/datasets/

frames = sorted(glob.glob('/content/datasets/mall_dataset/frames/*.jpg'))
# Read the frame and get its dimensions
base_frame = cv2.imread(frames[FRAME_INDEX])
H, W, _ = base_frame.shape

# --- 2. Setup Video Writer ---
# Create a VideoWriter object to save the video
out = cv2.VideoWriter(OUTPUT_FILE, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (W, H))

print(f"Generating incident response video for '{LOCATION_NAME}'...")

# --- 3. Create the Alert Frame ---
# Draw the emergency bounding box around the person
x1, y1, x2, y2 = EMERGENCY_BOX
cv2.rectangle(base_frame, (x1, y1), (x2, y2), (0, 0, 255), 3) # Thick red box

# --- Create the Pop-up Alert ---
# Increased width to 550 to fit the text
popup_w, popup_h = 550, 150
popup_x, popup_y = (W - popup_w) // 2, 50

# Draw popup background (white with red border)
cv2.rectangle(base_frame, (popup_x, popup_y), (popup_x + popup_w, popup_y + popup_h), (255, 255, 255), -1)
cv2.rectangle(base_frame, (popup_x, popup_y), (popup_x + popup_w, popup_y + popup_h), (0, 0, 255), 3)

# Add Title with Icon (Red Cross)
# Adjusted x-coordinates slightly for better centering in the wider box
cv2.putText(base_frame, "+", (popup_x + 20, popup_y + 50), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
cv2.putText(base_frame, INCIDENT_TYPE, (popup_x + 80, popup_y + 45), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 3)

# Add Location and Action details
cv2.putText(base_frame, f"Location: {LOCATION_NAME}", (popup_x + 30, popup_y + 90), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (50, 50, 50), 2)
cv2.putText(base_frame, f"ACTION: {ACTION_PLAN}", (popup_x + 30, popup_y + 130), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

# --- 4. Write Frames to Video ---
# We repeat the same frame to create a static video showing the alert
for _ in range(DURATION_FRAMES):
    out.write(base_frame)

# Release the VideoWriter
out.release()
print(f"Video saved as {OUTPUT_FILE}")

# --- Download ---
# This will prompt your browser to download the generated video file
files.download(OUTPUT_FILE)

Generating incident response video for 'North Corridor - Bench Area B'...
Video saved as medical_emergency_response_fixed.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>